In [0]:
from pyspark.sql import SparkSession

# 1. Initialize Spark Session
spark = SparkSession.builder.appName("HighestSalaryPerGenderSQL").getOrCreate()

# 2. Create the Input Data and register it as a temporary view
data = [
    (1, "sara", "2k", "Female"),
    (2, "joh", "3k", "Female"),
    (3, "jeo", "5k", "Male"),
    (4, "cemo", "6k", "Male"),
    (5, "triv", "2k", "Female"),
    (6, "buch", "5k", "Male"),
    (7, "rev", "10k", "Female")
]

df = spark.createDataFrame(data, ["ID", "NAME", "SAL", "GENDER"])
df.createOrReplaceTempView("input_table")

print("--- Original Input Table ---")
spark.sql("SELECT * FROM input_table").show()

# 3. Spark SQL Query using FIRST_VALUE and windowing
df_new = spark.sql("""
    SELECT 
        ID, 
        NAME, 
        SAL, 
        GENDER AS GENDER_cat,
        
        -- Grabs the name of the highest earner per gender partition
        FIRST_VALUE(NAME) OVER (
            PARTITION BY GENDER 
            ORDER BY CAST(REPLACE(SAL, 'k', '') AS INT) DESC
        ) AS Higest_sal_name
        
    FROM input_table
    ORDER BY GENDER, ID
""")

display(df_new)

In [0]:
from pyspark.sql import SparkSession

# Initialize Spark session
spark = SparkSession.builder.appName("PerformanceTransformation").getOrCreate()

# Sample input data matching your interview question
data = [
    ("G2 Outperformed / L1 Role Model",),
    ("G2 highperformed / L3 consistence performance",)
]

# Create DataFrame
df = spark.createDataFrame(data, ["perf_id"])

# Register as a temporary view so we can use Spark SQL
df.createOrReplaceTempView("input_table")

df_new = spark.sql("""
    SELECT 
        -- 1. Construct perf_id
        CONCAT(
            REGEXP_SUBSTR(perf_id, 'G[0-9]+'), 
            REGEXP_SUBSTR(perf_id, 'L[0-9]+')
        ) AS perf_id,
        
        -- 2. Calculate perf_value
        CAST(REGEXP_REPLACE(REGEXP_SUBSTR(perf_id, 'G[0-9]+'), '[^0-9]', '') AS INT) + 
        CAST(REGEXP_REPLACE(REGEXP_SUBSTR(perf_id, 'L[0-9]+'), '[^0-9]', '') AS INT) AS perf_value

    FROM input_table
""")

display(df_new)

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("PerformanceTransformationPySpark").getOrCreate()

# Same input data as the Spark SQL version
data = [
    ("G2 Outperformed / L1 Role Model",),
    ("G2 highperformed / L3 consistence performance",)
]

df = spark.createDataFrame(data, ["perf_id"])

# Extract the G<number> and L<number> tokens natively
# regexp_extract(..., idx=0) returns the entire match, like REGEXP_SUBSTR does
g_match = F.regexp_extract("perf_id", r"G[0-9]+", 0)
l_match = F.regexp_extract("perf_id", r"L[0-9]+", 0)

# Strip non-digits, cast to int, and add -> perf_value
g_num = F.regexp_replace(g_match, "[^0-9]", "").cast("int")
l_num = F.regexp_replace(l_match, "[^0-9]", "").cast("int")

df_new = df.select(
    F.concat(g_match, l_match).alias("perf_id"),
    (g_num + l_num).alias("perf_value")
)

display(df_new)

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import expr

spark = SparkSession.builder.appName("AlternateCapitalize").getOrCreate()

data = [("hello world spark sql",), ("apache spark data engineering",)]
df = spark.createDataFrame(data, ["sentence"])

df_new = df.withColumn(
    "sentence",
    expr("""
        array_join(
            transform(
                split(sentence, ' '),
                (word, i) -> case when i % 2 = 0 then initcap(word) else lower(word) end
            ),
            ' '
        )
    """)
)

df_new.show(truncate=False)

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder.appName("PurePySparkAlternateCapitalize").getOrCreate()

# Sample data with messy/multiple spaces
data = [("hello   world  spark sql",), ("apache   spark   data   engineering",)]
df = spark.createDataFrame(data, ["sentence"])

df_new = df.withColumn(
    "sentence",
    F.array_join(
        F.transform(
            # Split using regex to handle multiple spaces safely
            F.split(F.col("sentence"), r"\s+"),
            # Lambda function: word = item, i = index (0-based)
            lambda word, i: F.when(i % 2 == 0, F.initcap(word)).otherwise(F.lower(word))
        ),
        " " # Join back with a single space
    )
)

df_new.show(truncate=False)

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

# Initialize Spark session
spark = SparkSession.builder.appName("WordCountEnhancedPySpark").getOrCreate()

# Deliberately messy data: nulls, empty strings, pure punctuation,
# tabs/newlines, multiple spaces, mixed case, unicode, stray numbers
sentences = [
    ("   Hello   world!!!   ",),                     # extra spaces + punctuation
    ("Spark@@ is  AWESOME... #spark @databricks",),  # special chars + mixed case
    ("\tword-count -- ??  \n",),                     # tab, newline, hyphens
    ("Hello WORLD hello world 123",),                # repeated words + number
    ("",),                                           # empty string
    ("!!!@@@$$%%^^&&**",),                           # punctuation only
    ("café Ünicode résumé",),                        # accented characters
    (None,),                                         # null
]

df = spark.createDataFrame(sentences, ["line"])

# Pipeline execution
word_counts = (
    df
    # 1. Handle nulls and standardize to lowercase
    .withColumn("line", F.coalesce(F.col("line"), F.lit("")))
    .withColumn("line", F.lower("line"))
    
    # 2. Replace anything that is not a letter, digit, or whitespace with a space
    .withColumn("line", F.regexp_replace("line", r"[^a-z0-9\s]", " "))
    
    # 3. Split by r"\s+" (handles multiple spaces, tabs, newlines safely)
    #    Explode array into individual rows, and drop empty tokens caused by outer boundaries/spacing
    .select(F.explode(F.split(F.col("line"), r"\s+")).alias("word"))
    .where(F.col("word") != "")
    
    # 4. Drop tokens with no letters (e.g., pure digits like '123')
    .where(F.col("word").rlike("[a-z]"))
    
    # 5. Group by unique words, count frequencies, and sort deterministically
    .groupBy("word")
    .count()
    .orderBy(F.desc("count"), F.col("word"))
)

# Display the clean output
word_counts.show(20, truncate=False)

In [0]:
from pyspark.sql import SparkSession

spark = SparkSession.builder.appName("WordCountBadDataSQL").getOrCreate()

# Same messy data as the PySpark version
sentences = [
    ("  Hello   world!!!  ",),
    ("Spark@@ is   AWESOME... #spark @databricks",),
    ("\tword-count -- ??  \n",),
    ("Hello WORLD hello world 123",),
    ("",),
    ("!!!@@@$$%%^^&&**",),
    ("café Ünicode résumé",),
    (None,),
]
df = spark.createDataFrame(sentences, ["line"])
df.createOrReplaceTempView("messy_table")

word_counts = spark.sql("""
    WITH cleaned AS (
        SELECT
            lower(coalesce(line, '')) AS line
        FROM messy_table
    ),
    normalized AS (
        SELECT
            -- replace anything that is not a letter/digit/space with a space
            -- (punctuation, tabs and newlines all become word separators)
            regexp_replace(line, '[^a-z0-9 ]', ' ') AS line
        FROM cleaned
    ),
    words AS (
        SELECT explode(split(line, ' ')) AS word
        FROM normalized
    )
    SELECT word, count(*) AS count
    FROM words
    -- drop empty tokens and tokens with no letters (pure digits)
    WHERE word != '' AND word rlike '[a-z]'
    GROUP BY word
    ORDER BY count DESC, word
""")

display(word_counts)